# Chapter 4 · Looking back: attention

Every model so far has seen one character, or two, before making its guess. To guess the
blank in "To be, or not to b_", you want to look much further back than that.

**Attention** lets each position look back over *all* the characters before it and decide
which ones matter. Every position asks a question (its *query*), every position
advertises what it holds (its *key*), and the best matches pass their information along
(their *value*). It needs no table of contexts, so the memory can be long.

| | |
|---|---|
| **Open box** | one attention head: `Head` in `tinygpt/model.py` |
| **The number** | 2.48 → 1.76 |
| **Time** | about an hour |
| **Python you need** | reading a class; a little PyTorch |
| **GPU** | recommended; on a CPU each training run takes a minute or two |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    repo = Path("/content/tinyGPT")
    if not repo.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(repo))
    root = repo / "entry"
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

You will train a small model with **one block** of attention (four heads side by side)
and 64 characters of memory, for about a minute.

In [ ]:
guess_loss = None       # its validation loss? (counting pairs: 2.48; two characters of memory: 2.05)
guess_no_mask = None    # the same model, but allowed to peek at the next character: its loss?

And in words: if a model can peek at the answer while it trains, will its *writing* be
better or worse?

*Your explanation (double-click to edit):*

## 2 · Run `everyone`

The model is the `small` config from `configs/small.yaml`, cut down to one block.

In [ ]:
import torch

tok, train_ids, val_ids = tinygpt.load_data()
cfg = tinygpt.load_config("configs/small.yaml", n_layer=1)
model, history = tinygpt.train(cfg)

In [ ]:
tinygpt.plot_history(history)
print("your guess:", guess_loss)

Let it write:

In [ ]:
print(tinygpt.sample(model, tok, prompt="ROMEO:", length=300, seed=0))

## 3 · Investigate `everyone`

### Open the box

One attention head. The model you just trained has four of them, side by side.

In [ ]:
tinygpt.show(tinygpt.model.Head)

1. `query`, `key` and `value` each turn a character's 128 numbers into 32. Which one is
   "what am I looking for?", which is "what do I contain?", and which is "what do I pass
   on?"
2. `scores = q @ k.transpose(-2, -1)` makes a T × T table. What does the number in row
   *t*, column *s* measure?
3. `torch.tril` keeps the lower triangle of a table (run the next cell to see one). Why
   must position *t* never look at position *t* + 1?
4. After the softmax, each row adds up to 1. What does `forward` then do with those
   weights and the values?

In [ ]:
print(torch.tril(torch.ones(5, 5, dtype=torch.int)))      # 1 = allowed to look, 0 = hidden

*Your explanation (double-click to edit):*

### Where does it look?

Each picture shows one head reading a line. Row *t* is the head reading character *t*;
the dark squares in that row are the earlier characters it looks at most.

In [ ]:
line = "ROMEO:\nBut, soft! what light through yonder window breaks?"
for head in range(cfg.n_head):
    tinygpt.attention_map(model, tok, line, head=head)

- Is one head mostly a thin line just below the diagonal? What is it looking at, and why
  would that help?
- Do other heads have vertical stripes at particular characters, such as spaces or
  capital letters? What might they be tracking?
- Nobody programmed these patterns. Where did they come from?

*Your explanation (double-click to edit):*

## 4 · Modify `most people`

**A. Take the mask away.** `causal=False` removes the `torch.tril` line's effect: every
position can now see the whole window, including the character it is meant to predict.
**Predict** the loss and the quality of the writing, then run.

In [ ]:
cheat_cfg = tinygpt.load_config("configs/small.yaml", n_layer=1, causal=False)
cheater, cheat_history = tinygpt.train(cheat_cfg)
tinygpt.plot_history({"with the mask": history, "without the mask": cheat_history})

In [ ]:
print(repr(tinygpt.sample(cheater, tok, prompt="ROMEO:", length=200, seed=0)))

An almost perfect loss, and gibberish: a character or two, repeated. While training, it
learned to copy the answer from the next position. When it writes, there is no next
position to copy from. A beautiful training curve, a useless model. Explain why the low
loss misled us:

*Your explanation (double-click to edit):*

**B. Memory length.** `block_size` is how many characters the model can look back over.
**Predict** the loss with 8 and 16 characters, then compare with your 64.

In [ ]:
memory = {"64 characters": history}
for size in [8, 16]:
    short_cfg = tinygpt.load_config("configs/small.yaml", n_layer=1, block_size=size)
    short_model, memory[f"{size} characters"] = tinygpt.train(short_cfg)
tinygpt.plot_history(memory)

Longer memory, lower number, and the model hardly grows: only the position table gets
bigger. In chapter 2, every extra character of memory made the counting table 65 times
bigger. How did attention avoid that?

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Write attention yourself: the four steps inside `Head`, as one function. The check tries
it on random tensors and compares the answer with tinyGPT's.

In [ ]:
import math
from torch.nn import functional as F

def my_attention(q, k, v):
    """Causal self-attention for one head. q, k and v are tensors of shape (B, T, head_size)."""
    T = q.shape[1]
    # 1. scores: q times k transposed, divided by math.sqrt(head_size)
    # 2. hide the future: set the scores above the diagonal to float("-inf")
    # 3. softmax over the last dimension, so each row sums to 1
    # 4. return the weighted average of the values
    return v                      # replace this line

tinygpt.check.attention(my_attention)

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/04_attention.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
tinygpt.scoreboard({"learning pairs (chapter 3)": 2.49, "one block of attention": history[-1]["val_loss"]})

## Common questions

**What are query, key and value, really?** Think of a library. Your *query* is what you
are looking for; each book's *key* is the label on its spine; its *value* is what is
inside. Attention compares your query with every label and reads mostly from the books
that match.

**Why divide by the square root of `head_size`?** The scores are sums of `head_size`
products, so they grow as heads get wider. Very large scores make the softmax put almost
all its weight on one position; scaling keeps it from becoming that sharp.

**Why four heads instead of one big one?** Each head can look for something different
in the same line, as the pictures showed.

**Is this the attention that ChatGPT uses?** The same mechanism: causal self-attention
with queries, keys and values. It has many more heads and layers, and much faster code,
but the four steps are the ones in `Head`.

## If something breaks

| What you see | What to do |
|---|---|
| `ValueError: got 70 characters, but block_size is 64` | The line given to `attention_map` is longer than the model's memory. Use a shorter line. |
| The check says every position can see the future | Mask the scores *before* the softmax, not after. |
| Training is slow | Each run is a minute or two on a CPU. In Colab, a GPU runtime makes it faster: Runtime > Change runtime type. |

## Next

**Chapter 5 · The full run.** Stack six bigger blocks and train the full tinyGPT,
10.7 million parameters, in under ten minutes. The number reaches 1.49.